In [3]:
import csv
import warnings

import seaborn as sns
import pandas as pd
import numpy as np
import matplotlib as mpl

from typing import List
from scipy import integrate
from datetime import datetime, time, timedelta
from matplotlib import pyplot as plt
from matplotlib import font_manager as fm


warnings.simplefilter(action='ignore', category=FutureWarning)
pd.options.display.float_format = '.{:.5f}'.format
sns.reset_defaults()
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Linux Libertine"],
    "font.size": 9,
    "text.usetex": True,
    "text.latex.preamble": r"\usepackage{libertine}",
    "mathtext.fontset": "custom",
    "mathtext.rm": "Linux Libertine",
    "mathtext.it": "Linux Libertine:italic",
    "mathtext.bf": "Linux Libertine:bold",
})

FIGSIZE_ACM_SMALL = (3.33, 2.06)
FIGSIZE_ACM_SMALL = (7.00, 2.06)

wearables = [[86, 166, 211], [26, 63, 110], [87, 132, 148], [224, 88, 108], [16, 80, 103]]
wearables_palette = [[r/255, g/255, b/255] for r, g, b in wearables ]
sns.set_palette(wearables_palette)

In [17]:
def get_csv_as_list(filename: str) -> list[list[str]]:
    values = []
    with open(filename) as f:
        reader = csv.reader(f)
        for row in reader:
            row[0] = row[0][20:]
            values.append(row)
    return values[1:]

def parse_csv(filename: str) -> pd.DataFrame:
    def parse(data: list[str]) -> pd.DataFrame:
            return pd.DataFrame(data, columns=["message_id", "t_produced", "t_observed"])
    data = get_csv_as_list(filename)
    return parse(data)

def transform_columns_to_numerics(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["message_id"] = pd.to_numeric(df["message_id"])
    df["t_produced"] = pd.to_numeric(df["t_produced"])
    df["t_observed"] = pd.to_numeric(df["t_observed"])
    return df

def filter_errors(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    

def reduce_frame(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    reduced_df = df.groupby(["message_id"], as_index=True).agg(
        t_produced=("t_produced", "min"), 
        t_observed=("t_observed", "max")
    )
    return reduced_df


def trim_benchmark(df: pd.DataFrame, ramp_up_s: int = 15, ramp_down_s: int = 5) -> pd.DataFrame:
    df = df.copy()
    start_time = df["t_produced"].min() + ramp_up_s * 1_000_000_000
    end_time = df["t_observed"].max() - ramp_down_s * 1_000_000_000

    trimmed = df[(df["t_produced"] >= start_time) & (df["t_observed"] <= end_time)]
    return trimmed

def median_processing_time(df: pd.DataFrame) -> float:
    df = df.copy()
    df["processing_time"] = df["t_observed"] - df["t_produced"]
    median_pt = df["processing_time"].median()
    return median_pt

def convert_to_sec(nano: float) -> float:
    return nano / (1_000_000_000)

def process(filename: str) -> float:
    csv = parse_csv(filename)
    transformed = transform_columns_to_numerics(csv)
    trimmed = trim_benchmark(transformed)
    reduced = reduce_frame(trimmed)
    return convert_to_sec(median_processing_time(reduced))

def average_median(m1: float, m2: float, m3: float) -> float:
    avg = (m1 + m2 + m3) / 3
    return avg

In [19]:
# Mapper

# 1KB, 1RPS, 60s
median01_1kb = process("measurements/mapper/1kb_1RPS_60s_01.csv")
median02_1kb = process("measurements/mapper/1kb_1RPS_60s_02.csv")
median03_1kb = process("measurements/mapper/1kb_1RPS_60s_03.csv")


# 10 KB, 1RPS, 60s
median01_10kb = process("measurements/mapper/10kb_1RPS_60s_01.csv")
median02_10kb = process("measurements/mapper/10kb_1RPS_60s_02.csv")
median03_10kb = process("measurements/mapper/10kb_1RPS_60s_03.csv")

# 50 KB, 1RPS, 60s
median01_50kb = process("measurements/mapper/50kb_1RPS_60s_01.csv")
median02_50kb = process("measurements/mapper/50kb_1RPS_60s_02.csv")
median03_50kb = process("measurements/mapper/50kb_1RPS_60s_03.csv")

# 25KB, 1RPS, 60s
median01_25kb = process("measurements/mapper/25kb_1RPS_60s_01.csv")
median02_25kb = process("measurements/mapper/25kb_1RPS_60s_02.csv")
median03_25kb = process("measurements/mapper/25kb_1RPS_60s_03.csv")
print(median01_25kb)
print(median02_25kb)
print(median03_25kb)

print("mapper 1kb:")
print(f"{average_median(median01_1kb, median02_1kb, median03_1kb)}s")
print("mapper 10kb:")
print(f"{average_median(median01_10kb, median02_10kb, median03_10kb)}s")
print("mapper 25kb:")
print(f"{average_median(median01_25kb, median02_25kb, median03_25kb)}s")
print("mapper 50kb:")
print(f"{average_median(median01_50kb, median02_50kb, median03_50kb)}s")



10.385248601
11.6303206935
12.3527254085
mapper 1kb:
0.21163149299999998s
mapper 10kb:
1.560024951s
mapper 25kb:
11.456098234333334s
mapper 50kb:
22.483608742166666s


In [21]:
# RPS effects on processing time:


plot3r = pd.DataFrame ({
    "Requests/Second": [1, 2, 3],
    "Median Latency": [34.336, 36.22, 38.333] # TODO -> Test values
})

plot5r = pd.DataFrame ({
    "Requests/Second": [1, 2, 3],
    "Median Latency": [32.12, 33.5, 35.99] 
})

plot7r = pd.DataFrame ({
    "Requests/Second": [1, 2, 3],
    "Median Latency": [31.00, 32.8, 35.7] 
})

df = pd.concat([
    plot3r.assign(system="3 Replicas"),
    plot5r.assign(system="5 Replicas"),
    plot7r.assign(system="7 Replicas"),
])

fig, ax = plt.subplots(figsize=FIGSIZE_ACM_SMALL, constrained_layout=True)

sns.lineplot(
    data=df,
    x="Requests/Second",
    y="Median Latency",
    hue="system",
    style="system",
    markers=True,
    dashes=True,
    ax=ax
)

ax.set_ylabel("Median Latency [s]")
ax.set_xlabel("Requests/Second")
ax.set_ylim(30, 45)
ax.set_xlim(0.5, 3.5)
ax.set_xticks([1,2,3])
ax.set_xticklabels(["$1$", "$10$", "$100$"])

handles, labels = ax.get_legend_handles_labels()
ax.legend(handles=handles, labels=labels, loc="center left")

fig.savefig("plots/rps_impact_on_processing_time.pdf", bbox_inches=False)
plt.close()

In [23]:
# Message-Size Impact on Processing Time

plot = pd.DataFrame({
    "MessageSize": [1, 2, 3, 4],
    "ProcessingTime": [0.21163149299999998, 1.560024951, 11.456098234333334, 22.483608742166666],
}).assign(Replicas="3")


fig, ax = plt.subplots(figsize=FIGSIZE_ACM_SMALL, constrained_layout=True)

sns.lineplot(
    data=plot,
    x="MessageSize",
    y="ProcessingTime",
    hue="Replicas",
    style="Replicas",
    markers=True,
    dashes=True,
    ax=ax
)

ax.set_ylabel("Processing Time [s]")
ax.set_xlabel("Message Size [KB]")
ax.set_ylim(0, 30)
ax.set_xlim(0.5, 4.5)
ax.set_xticks([1,2,3,4])
ax.set_xticklabels(["$1$", "$10$", "$25$","$50$"])

fig.savefig("plots/message-size-impact.pdf", bbox_inches=False)
plt.close()

In [117]:
# Ingestion-Service

# What do I want to show?
# Has Message-Size an impact on Response Time?
# Where is the Breakpoint? -> For 10KB Messages (not tested yet)

# 1KB messages

def parse_csv_ingestion_service(filename: str) -> pd.DataFrame:
    def parse(data: list[str]) -> pd.DataFrame:
        return pd.DataFrame(data, columns=["message_id", "status_code", "t_send", "t_received"])
    data = get_csv_as_list(filename)
    return parse(data)

def transform_columns_to_numerics_is(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["message_id"] = pd.to_numeric(df["message_id"])
    df["status_code"] = pd.to_numeric(df["status_code"])
    df["t_send"] = pd.to_numeric(df["t_send"])
    df["t_received"] = pd.to_numeric(df["t_received"])
    return df

def trim_benchmark_is(df: pd.DataFrame, ramp_up_s: int = 15, ramp_down_s: int = 5) -> pd.DataFrame:
    df = df.copy()

    df["t_send"] = pd.to_numeric(df["t_send"])
    
    min_time = df['t_send'].min()
    max_time = df['t_send'].max()
    
    start_cutoff = min_time + ramp_up_s * 1_000_000_000
    end_cutoff = max_time - ramp_down_s * 1_000_000_000
    
    trimmed_df = df[
        (df['t_send'] >= start_cutoff) & 
        (df['t_send'] <= end_cutoff)
    ]
    
    return trimmed_df


def calculate_error_rate(df: pd.DataFrame) -> float:
    df = df.copy()

    error_count = (df["status_code"] != 200).sum()
    print(f"counted: {error_count} errors")

    total_rows = len(df)
    error_rate = error_count / total_rows if total_rows > 0 else 0
    
    return error_rate

def ingestion_service_benchmark_metrics(df: pd.DataFrame) -> dict:
    df["response_time"] = df["t_received"] - df["t_send"]
    
    successful = df[df["status_code"] == 200]
    errors = df[df["status_code"] != 200]

    median_response_time = ns_to_ms(successful["response_time"].median())
    p95_response_time = ns_to_ms(successful["response_time"].quantile(0.95))
    p99_response_time = ns_to_ms(successful["response_time"].quantile(0.99))
    error_rate = len(errors) / len(df) if len(df) > 0 else 0
    median_error_response_time =  ns_to_ms(errors["response_time"].median())


    print(f"median_response_time: {median_response_time}ms")
    print(f"p95_response_time: {p95_response_time}ms")
    print(f"p99_response_time: {p99_response_time}ms")
    print(f"error_rate: {error_rate}")
    if error_rate > 0:
        print(f"median_error_response_time: {median_error_response_time}ms")

def median_response_time_overtime(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["t_received"] = pd.to_numeric(df["t_received"])
    df["t_send"] = pd.to_numeric(df["t_send"])

    df["response_time"] = df["t_received"] - df["t_send"]

    t0 = df["t_received"].min()
    df["second"] = (df["t_received"] - t0) // 1_000_000_000

    ok = df[df["status_code"] == 200]
    result = (
        ok.groupby("second")["response_time"]
          .median()
          .reset_index(name="median_response_time_ns")
    )
    result["median_response_time_ms"] = result["median_response_time_ns"].apply(ns_to_ms)

    result["second"] = result["second"].astype(int)
    
    return result[["second", "median_response_time_ms"]]


def throughput_error_rate_per_s(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["t_received"] = pd.to_numeric(df["t_received"])
    
    t0 = df["t_received"].min()
    df["second"] = (df["t_received"] - t0) // 1_000_000_000

    ok = df[df["status_code"] == 200]
    err = df[df["status_code"] != 200]

    throughput = (
        ok.groupby("second")
          .size()
          .reset_index(name="throughput")
    )

    errs = (
        err.groupby("second")
           .size()
           .reset_index(name="errors")
    )

    result = pd.merge(throughput, errs, on="second", how="outer").fillna(0)

    result["throughput"] = result["throughput"].astype(int)
    result["errors"] = result["errors"].astype(int)

    return result.sort_values("second").reset_index(drop=True)


def process_ingestion_service(filename: str):
    df = parse_csv_ingestion_service(filename)
    df = trim_benchmark_is(df)
    df = transform_columns_to_numerics_is(df)
    # ingestion_service_benchmark_metrics(df)
    return throughput_error_rate_per_s(df)
    # return median_response_time_overtime(df)

def process_response_time(filename: str):
    df = parse_csv_ingestion_service(filename)
    df = trim_benchmark_is(df)
    df = transform_columns_to_numerics_is(df)
    return median_response_time_overtime(df)


def error_rate(filename: str) -> float:
    df = parse_csv_ingestion_service(filename)
    df = transform_columns_to_numerics_is(df)
    return calculate_error_rate(df)

def avg_of_columns(df1, df2, df3):
    return pd.concat([df1, df2, df3]).groupby(level=0).mean()
    

def ns_to_ms(nano: int) -> float:
    return nano / (1_000_000)

# process_ingestion_service("measurements/ingestion-service/1kb_1RPS_60s_01.csv")
# process_ingestion_service("measurements/ingestion-service/1kb_1RPS_60s_02.csv")
# process_ingestion_service("measurements/ingestion-service/1kb_1RPS_60s_03.csv")
print("50KB, 50RPS, 60s")
df1 = process_ingestion_service("measurements/ingestion-service/50kb_50RPS_60s_01.csv")
print("---")
df2 = process_ingestion_service("measurements/ingestion-service/50kb_50RPS_60s_02.csv")
print("---")
df3 = process_ingestion_service("measurements/ingestion-service/50kb_50RPS_60s_03.csv")

avg = avg_of_columns(df1, df2, df3)

# Plot
# avg has 2 curves

plotThroughput = avg[["second", "throughput"]].rename(columns={"throughput": "value"})
plotThroughput["Metric"] = "Throughput/s"

plotErrors = avg[["second", "errors"]].rename(columns={"errors": "value"})
plotErrors["Metric"] = "Errors/s"

plot = pd.concat([plotThroughput, plotErrors], ignore_index=True)

fig, ax = plt.subplots(figsize=FIGSIZE_ACM_SMALL, constrained_layout=True)

sns.lineplot(
    data=plot,
    x="second",
    y="value",
    hue="Metric",
    style="Metric",
    markers=True,
    dashes=True,
    ax=ax
)

ax.set_xlabel("Time [s]")
ax.set_ylabel("Count")
ax.set_ylim(0, max(plot["value"]) * 1.1)

fig.savefig("plots/ingestion-service-throughput-errors.pdf", bbox_inches=False)
plt.close()


# plot response time
df1 = process_response_time("measurements/ingestion-service/50kb_50RPS_60s_01.csv")
df2 = process_response_time("measurements/ingestion-service/50kb_50RPS_60s_02.csv")
df3 = process_response_time("measurements/ingestion-service/50kb_50RPS_60s_03.csv")

avg = avg_of_columns(df1, df2, df3)

plotResponseTime = avg[["second", "median_response_time_ms"]].rename(columns={"median_response_time_ms": "value"}).assign(Metric="Response Time")

fig, ax = plt.subplots(figsize=FIGSIZE_ACM_SMALL, constrained_layout=True)
sns.lineplot(
    data=plotResponseTime,
    x="second",
    y="value",
    hue="Metric",
    style="Metric",
    markers=True,
    dashes=True,
    ax=ax
)

ax.set_xlabel("Time [s]")
ax.set_ylabel("Median Response Time [ms]")
ax.set_ylim(0, 10)

fig.savefig("plots/ingestion-service-response-time.pdf", bbox_inches=False)
plt.close()


50KB, 50RPS, 60s
---
---


In [ ]:
# Plotting the Ingestion-Service

# message-size impact